# Neighbourhood Heat-Risk Communication & Response Planner
## Experiment Notebook
**Purpose:** Compare baseline generic advisory strategy vs neighbourhood-targeted strategy.

> **Note:** This notebook uses only aggregate neighbourhood-level data. No personally identifiable information is used.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('neighbourhood_heat_risk.csv')
print(f'Dataset: {len(df)} neighbourhoods')
df.head()

## 1. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Risk Level Distribution ===')
print(df['risk_level'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Indicator Distribution by Risk Level', fontsize=14, fontweight='bold')

colors = {'High': '#ef4444', 'Medium': '#f97316', 'Low': '#22c55e', 'AMBIGUOUS': '#a855f7'}
valid = df[df['risk_level'].isin(['High', 'Medium', 'Low'])].copy()

indicators = [
    ('temperature', 'Temperature (°C)'),
    ('vulnerability_score', 'Vulnerability Score'),
    ('built_environment_score', 'Built Env. Score'),
    ('service_access_score', 'Service Access Score'),
    ('elderly_percentage', 'Elderly %'),
    ('risk_score', 'Risk Score'),
]

for ax, (col, label) in zip(axes.flat, indicators):
    for level in ['High', 'Medium', 'Low']:
        subset = valid[valid['risk_level'] == level][col].dropna()
        ax.hist(subset, alpha=0.65, bins=12, color=colors[level], label=level)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: eda_distributions.png')

## 2. Indicator Correlation Analysis

In [ ]:
indicators_numeric = [
    'temperature', 'humidity', 'heat_index',
    'elderly_percentage', 'low_income_percentage',
    'vulnerability_score', 'built_environment_score',
    'service_access_score', 'risk_score'
]
corr = df[indicators_numeric].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)
ax.set_title('Indicator Correlation Matrix', fontsize=13, fontweight='bold')

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='black' if abs(corr.iloc[i,j]) < 0.7 else 'white')

plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Transparent Risk Score Calculation

In [ ]:
WEIGHTS = {'climate': 0.35, 'vulnerability': 0.30, 'built_env': 0.20, 'service_risk': 0.15}

def normalize_col(series, lo=None, hi=None):
    lo = series.min() if lo is None else lo
    hi = series.max() if hi is None else hi
    return (series - lo) / (hi - lo + 1e-9)

calc = df.copy()

# Use heat_index where available, else temperature
calc['climate_val'] = calc['heat_index'].fillna(calc['temperature'])
calc['climate_norm']  = normalize_col(calc['climate_val'])
calc['vuln_norm']     = normalize_col(calc['vulnerability_score'])
calc['built_norm']    = normalize_col(calc['built_environment_score'])
calc['serv_risk']     = 1 - normalize_col(calc['service_access_score'])

calc['computed_score'] = (
    calc['climate_norm']  * WEIGHTS['climate'] +
    calc['vuln_norm']     * WEIGHTS['vulnerability'] +
    calc['built_norm']    * WEIGHTS['built_env'] +
    calc['serv_risk']     * WEIGHTS['service_risk']
) * 100

# Compare with stored scores
valid_both = calc[calc['risk_score'].notna() & calc['computed_score'].notna()]
correlation = valid_both['risk_score'].corr(valid_both['computed_score'])
print(f'Correlation between stored and computed risk scores: {correlation:.3f}')

fig, ax = plt.subplots(figsize=(7, 6))
level_colors = valid_both['risk_level'].map({'High': '#ef4444', 'Medium': '#f97316', 'Low': '#22c55e'})
ax.scatter(valid_both['risk_score'], valid_both['computed_score'],
           c=level_colors, alpha=0.7, s=60, edgecolors='white', linewidth=0.5)
ax.plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect match')
ax.set_xlabel('Stored Risk Score (from CSV)', fontsize=11)
ax.set_ylabel('Recomputed Risk Score (notebook)', fontsize=11)
ax.set_title(f'Risk Score Reproducibility (r={correlation:.3f})', fontsize=12, fontweight='bold')
patches = [mpatches.Patch(color=c, label=l) for l, c in [('High','#ef4444'),('Medium','#f97316'),('Low','#22c55e')]]
ax.legend(handles=patches, fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('score_reproducibility.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Baseline Strategy — Generic City-Wide Advisory

In [ ]:
# Baseline: all neighbourhoods treated equally, estimated 50% reach for high-risk
valid = df[df['risk_score'].notna()].copy()
high_risk = valid[valid['risk_level'] == 'High']
total = len(valid)

baseline_metrics = {
    'total_neighbourhoods':   total,
    'high_risk_count':        len(high_risk),
    'high_reached_pct':       50.0,
    'missed_high_risk':       round(len(high_risk) * 0.5),
    'precision':              50.0,
    'recall':                 50.0,
    'resource_util_pct':      40.0,
    'timeliness':             '3-5 days'
}

print('=== Baseline Strategy Metrics ===')
for k, v in baseline_metrics.items():
    print(f'  {k}: {v}')

## 5. Proposed Strategy — Neighbourhood-Targeted

In [ ]:
# Proposed: neighbourhood-specific targeting based on risk score
avg_reach_high = high_risk['communication_reach_percentage'].mean()

proposed_metrics = {
    'total_neighbourhoods':   total,
    'high_risk_count':        len(high_risk),
    'high_reached_pct':       round(avg_reach_high, 1),
    'missed_high_risk':       round(len(high_risk) * (1 - avg_reach_high / 100)),
    'precision':              round(len(high_risk) / total * 100, 1),
    'recall':                 round(avg_reach_high, 1),
    'resource_util_pct':      min(95.0, round(len(high_risk) / total * 100 + 40, 1)),
    'timeliness':             'Same day'
}

print('=== Proposed Strategy Metrics ===')
for k, v in proposed_metrics.items():
    print(f'  {k}: {v}')

## 6. Strategy Comparison — Objectives A and B

In [ ]:
MAX_RESOURCES = 20
priority_map = {'High': 3, 'Medium': 2, 'Low': 1}
valid_s = valid.copy()
valid_s['priority'] = valid_s['risk_level'].map(priority_map).fillna(0)

# Objective A: Maximum Coverage
obj_a = valid_s.sort_values(['priority', 'risk_score'], ascending=False).head(MAX_RESOURCES)
obj_a_high = obj_a[obj_a['risk_level'] == 'High']

# Objective B: Resource Efficiency (high risk only, top half resources)
obj_b = valid_s[valid_s['risk_level'] == 'High'].sort_values('risk_score', ascending=False).head(MAX_RESOURCES // 2)

comparison = pd.DataFrame({
    'Metric': ['Areas Reached', 'High-Risk Coverage %', 'Est. Response Hours',
               'Resource Usage %', 'Urgent Areas Missed', 'Avg Risk Score'],
    'Obj A (Max Coverage)': [
        len(obj_a),
        round(len(obj_a_high) / max(len(high_risk), 1) * 100, 1),
        round(len(obj_a) * 1.5, 1),
        round(len(obj_a) / MAX_RESOURCES * 100, 1),
        len(high_risk) - len(obj_a_high),
        round(obj_a['risk_score'].mean(), 1),
    ],
    'Obj B (Resource Eff.)': [
        len(obj_b),
        round(len(obj_b) / max(len(high_risk), 1) * 100, 1),
        round(len(obj_b) * 1.5, 1),
        round(len(obj_b) / MAX_RESOURCES * 100 * 2, 1),
        len(high_risk) - len(obj_b),
        round(obj_b['risk_score'].mean(), 1),
    ],
})
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline vs Proposed bar chart
metrics_labels = ['High Reached %', 'Precision %', 'Recall %', 'Resource Util %']
baseline_vals  = [baseline_metrics['high_reached_pct'], baseline_metrics['precision'],
                  baseline_metrics['recall'], baseline_metrics['resource_util_pct']]
proposed_vals  = [proposed_metrics['high_reached_pct'], proposed_metrics['precision'],
                  proposed_metrics['recall'], proposed_metrics['resource_util_pct']]

x = np.arange(len(metrics_labels))
axes[0].bar(x - 0.2, baseline_vals, 0.35, label='Baseline', color='#94a3b8')
axes[0].bar(x + 0.2, proposed_vals, 0.35, label='Proposed', color='#1a56db')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_labels, fontsize=9)
axes[0].set_ylabel('Value (%)')
axes[0].set_title('Baseline vs Proposed Strategy', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Objective comparison
obj_metrics = ['Areas Reached', 'High-Risk %', 'Urgent Missed', 'Avg Score']
obj_a_vals  = [len(obj_a), round(len(obj_a_high)/max(len(high_risk),1)*100,1),
               len(high_risk)-len(obj_a_high), round(obj_a['risk_score'].mean(),1)]
obj_b_vals  = [len(obj_b), round(len(obj_b)/max(len(high_risk),1)*100,1),
               len(high_risk)-len(obj_b), round(obj_b['risk_score'].mean(),1)]

x2 = np.arange(len(obj_metrics))
axes[1].bar(x2 - 0.2, obj_a_vals, 0.35, label='Obj A: Max Coverage', color='#3b82f6')
axes[1].bar(x2 + 0.2, obj_b_vals, 0.35, label='Obj B: Resource Eff.', color='#22c55e')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(obj_metrics, fontsize=9)
axes[1].set_title('Objective A vs Objective B', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Measured Results Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'High-risk areas correctly prioritised',
        'High-risk areas reached (%)',
        'Communication timeliness',
        'Resource utilisation (%)',
        'Missed high-risk areas',
        'Precision (%)',
        'Recall / Coverage (%)',
    ],
    'Baseline': [
        baseline_metrics['high_risk_count'],
        f"{baseline_metrics['high_reached_pct']}%",
        baseline_metrics['timeliness'],
        f"{baseline_metrics['resource_util_pct']}%",
        baseline_metrics['missed_high_risk'],
        f"{baseline_metrics['precision']}%",
        f"{baseline_metrics['recall']}%",
    ],
    'Proposed': [
        proposed_metrics['high_risk_count'],
        f"{proposed_metrics['high_reached_pct']}%",
        proposed_metrics['timeliness'],
        f"{proposed_metrics['resource_util_pct']}%",
        proposed_metrics['missed_high_risk'],
        f"{proposed_metrics['precision']}%",
        f"{proposed_metrics['recall']}%",
    ],
})
print(summary.to_string(index=False))

## 8. Error Analysis

In [ ]:
missing_temp  = df['temperature'].isna().sum()
ambiguous     = (df['risk_level'] == 'AMBIGUOUS').sum()
extreme_vals  = (df['temperature'].fillna(0) > 52).sum()
no_cooling    = ((df['cooling_centres'] == 0) & (df['healthcare_facilities'] == 0)).sum()

print('=== Edge Cases & Failure Modes in Dataset ===')
print(f'  Neighbourhoods with missing temperature:          {missing_temp}')
print(f'  Ambiguous indicator cases:                        {ambiguous}')
print(f'  Extreme temperature outliers (>52°C):             {extreme_vals}')
print(f'  No cooling centres AND no healthcare facilities:  {no_cooling}')
print(f'\n  Total excluded from automated scoring:           {missing_temp + ambiguous + extreme_vals}')
print(f'  Percentage handled by human review:              {round((missing_temp + ambiguous + extreme_vals) / len(df) * 100, 1)}%')